# 데이터 분석 실습 노트북 (data_collection_worksapce 정리본)

이 노트북은 `sk-encore/data_collection_worksapce` 폴더의 여러 실습 노트북(dataNN_*.ipynb)에서
배운 내용 중, 웹 학습앱(`ml_study_playground.html`)에 새로 추가한 파트를 **진짜 파이썬 코드로
그대로 실습**할 수 있게 정리한 것입니다.

## 만든 과정 (어떻게 정리했는지)
1. `data_collection_worksapce` 폴더의 노트북들을 데이터 번호(dataNN) 순서로 훑어보며
   분류 알고리즘 / 앙상블 / 불균형 데이터 처리 / 군집화 / 통계 / 추천 시스템 / 파이썬 기초
   주제로 묶었습니다.
2. 웹 학습앱에는 "초등학생도 이해할 수 있는 쉬운 설명 + 숫자를 조절하며 실습하는 인터랙티브
   놀이터"를 자바스크립트로 만들었는데, 이 노트북에서는 **같은 계산을 pandas / scikit-learn /
   scipy로 그대로 재현**해서 진짜 파이썬 코드로도 확인할 수 있게 만들었습니다.
3. 각 셀마다 "왜 이 코드를 쓰는지"를 주석으로 남겨서, 나중에 다시 봐도 이해할 수 있게 정리했습니다.
4. `imbalanced-learn`처럼 별도 설치가 필요한 라이브러리는 셀 맨 위에 pip install 안내를
   주석으로 남겨두었습니다.

## 목차
1. 결정트리 — 지니 불순도로 첫 질문 고르기
2. 분류 알고리즘 비교 (KNN·결정트리·랜덤포레스트·SVM·보팅)
3. 불균형 데이터 처리 (SMOTE·오버샘플링·언더샘플링)
4. 군집화 (K-means)
5. 통계로 분포 모양 읽기 (평균·분산·왜도·첨도)
6. 영화 추천 시스템 (협업 필터링)
7. 파이썬 기초 — 튜플(tuple)


In [ ]:
# 이 노트북 전체에서 공통으로 쓰는 라이브러리를 먼저 불러옴
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트가 깨지지 않도록 설정 (윈도우는 맑은 고딕, 맥은 애플 고딕)
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지


## 1. 결정트리 — 지니 불순도로 첫 질문 고르기

결정트리는 스무고개처럼 "이 조건을 만족하나요?"라는 질문을 반복하면서 정답을 좁혀갑니다.
어떤 질문을 먼저 할지는 **지니 불순도(Gini Impurity)**를 가장 많이 줄여주는 질문을 골라서 정해요.

- 지니 불순도 = 1 - (스팸일 확률² + 정상일 확률²) → 0이면 완전히 순수(한 종류만 있음), 0.5면 가장 뒤섞임
- 정보이득(Information Gain) = 나누기 전 지니 - 나눈 후 가중평균 지니 → 클수록 좋은 질문

아래 스팸메일 10통 데이터(웹앱의 지니 계산기와 동일한 데이터)로, "free/win/call" 세 가지 질문 중
어떤 질문이 스팸을 가장 잘 구분하는지 직접 계산해봅니다.


In [ ]:
# 스팸메일 10통: free/win/call 단어 포함 여부(1=있음,0=없음)와 실제 스팸 여부(1=스팸,0=정상)
emails = pd.DataFrame({
    'free': [1,1,1,1,0,0,0,1,0,1],
    'win':  [1,0,0,0,0,0,0,0,0,1],
    'call': [1,1,1,0,0,0,0,0,1,1],
    'label':[1,1,1,1,0,0,0,0,0,1],
})

def gini(labels):
    """label 배열 하나를 받아 지니 불순도를 계산"""
    if len(labels) == 0:
        return 0
    p = labels.mean()  # 스팸(1)일 비율
    return 1 - (p**2 + (1-p)**2)

def info_gain(df, feature):
    """feature(0/1) 기준으로 나눴을 때 정보이득을 계산"""
    parent_gini = gini(df['label'])
    left = df[df[feature] == 1]['label']   # 그 단어가 있는 메일들
    right = df[df[feature] == 0]['label']  # 그 단어가 없는 메일들
    n = len(df)
    weighted_gini = (len(left)/n)*gini(left) + (len(right)/n)*gini(right)
    return parent_gini - weighted_gini

# 세 후보 질문(feature)의 정보이득을 각각 계산해서 비교
for feature in ['free', 'win', 'call']:
    ig = info_gain(emails, feature)
    print(f"'{feature}' 있나? 로 나눴을 때 정보이득 = {ig:.3f}")

best_feature = max(['free', 'win', 'call'], key=lambda f: info_gain(emails, f))
print(f"\n결정트리라면 가장 먼저 '{best_feature}가 있나?'를 물어볼 거예요 (정보이득이 가장 크니까)")


## 2. 분류 알고리즘 비교 — KNN · 결정트리 · 랜덤포레스트 · SVM · 보팅

분류 문제를 푸는 방법은 한 가지가 아니에요. 같은 데이터로 여러 알고리즘을 각각 학습시켜서
정확도를 비교해봅니다. (scikit-learn이 만들어주는 가상의 분류 데이터를 사용)


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 200개 샘플, 2개 클래스짜리 가상 분류 데이터 생성
X, y = make_classification(n_samples=200, n_features=6, n_informative=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# KNN과 SVM은 거리를 쓰기 때문에 스케일링을 먼저 해야 함 (Train 기준으로만 fit!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'KNN (K=5)':     KNeighborsClassifier(n_neighbors=5),
    '결정트리(Gini)':   DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42),
    '랜덤포레스트':      RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF 커널)':  SVC(kernel='rbf', probability=True, random_state=42),
}

results = {}
for name, model in models.items():
    # 거리 기반 모델(KNN, SVM)은 스케일된 데이터, 트리 기반 모델은 원본 데이터 사용
    if 'KNN' in name or 'SVM' in name:
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, pred)

# 보팅(Voting) 앙상블: 여러 모델의 확률 평균으로 최종 결정 (soft voting)
voting = VotingClassifier(estimators=[
    ('lr', LogisticRegression(max_iter=1000)),
    ('dt', DecisionTreeClassifier(max_depth=4, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
], voting='soft')
voting.fit(X_train, y_train)
results['보팅(soft)'] = accuracy_score(y_test, voting.predict(X_test))

for name, acc in results.items():
    print(f"{name:15s} 정확도 = {acc:.3f}")


## 3. 불균형 데이터 처리 — SMOTE · 오버샘플링 · 언더샘플링

이 셀은 `imbalanced-learn` 패키지가 필요합니다. 설치가 안 되어 있다면 먼저 실행하세요:
`!pip install imbalanced-learn`

정상 900개, 스팸 100개처럼 한쪽이 훨씬 적은 데이터를 그대로 학습하면, 모델이 "무조건 정상"이라고만
찍어도 정확도 90%가 나와버려서 정작 중요한 스팸을 하나도 못 잡아낼 수 있어요.


In [ ]:
# !pip install imbalanced-learn   # 처음 실행이면 주석을 풀고 설치하세요

from collections import Counter
from sklearn.datasets import make_classification

# 클래스 0이 90%, 클래스 1(소수, 예: 스팸)이 10%인 불균형 데이터 생성
X, y = make_classification(
    n_samples=1000, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, weights=[0.9, 0.1], flip_y=0, random_state=42
)
print("원본 클래스 분포:", Counter(y))

try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler

    # SMOTE: 소수 클래스 데이터 사이를 선으로 이어 그 위에 새 점을 생성 (단순 복제보다 똑똑함)
    X_smote, y_smote = SMOTE(random_state=42).fit_resample(X, y)
    print("SMOTE 후:             ", Counter(y_smote))

    # RandomOverSampler: 소수 클래스를 무작위로 복제
    X_ros, y_ros = RandomOverSampler(random_state=42).fit_resample(X, y)
    print("RandomOverSampler 후: ", Counter(y_ros))

    # RandomUnderSampler: 다수 클래스를 무작위로 줄임
    X_rus, y_rus = RandomUnderSampler(random_state=42).fit_resample(X, y)
    print("RandomUnderSampler 후:", Counter(y_rus))
except ImportError:
    print("imbalanced-learn이 설치되어 있지 않아요. 위 pip install 주석을 풀고 설치해보세요!")


## 4. 군집화 — K-means로 비슷한 것끼리 묶기

지금까지는 정답(label)이 있는 데이터로 배웠다면, K-means는 정답 없이 데이터끼리의 거리만 보고
스스로 무리를 찾아내는 **비지도학습**이에요. "중심점 찍기 → 가까운 점 모으기 → 중심점 옮기기"를
더 이상 안 바뀔 때까지 반복해요.


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# 3개의 무리로 흩어진 가상 2차원 데이터 생성 (진짜 정답은 안 알려주고 학습시켜볼 거예요)
X_blob, true_labels = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)

# 엘보우 방법: K를 1~6까지 바꿔가며 inertia(중심점과의 거리 제곱합)가 어떻게 줄어드는지 확인
inertias = []
for k in range(1, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_blob)
    inertias.append(km.inertia_)

plt.figure(figsize=(5, 3))
plt.plot(range(1, 7), inertias, marker='o')
plt.xlabel('K (군집 개수)')
plt.ylabel('inertia (중심점까지 거리 제곱합)')
plt.title('엘보우 방법 - 꺾이는 지점이 적당한 K')
plt.show()

# 꺾이는 지점(K=3)으로 실제 군집화 실행
km3 = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = km3.fit_predict(X_blob)

plt.figure(figsize=(5, 4))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=cluster_labels, cmap='viridis', s=25)
plt.scatter(km3.cluster_centers_[:, 0], km3.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='중심점')
plt.legend()
plt.title('K=3으로 군집화한 결과')
plt.show()


## 5. 통계로 분포 모양 읽기 — 평균 · 분산 · 왜도 · 첨도

데이터가 어떤 모양으로 퍼져있는지 그래프 없이도 숫자 4개로 요약할 수 있어요. 이 4가지를 통틀어
"적률(moment)"이라고 불러요. (`적률.ipynb` 원본 실습을 그대로 재현)


In [ ]:
from scipy import stats

np.random.seed(42)
normal_data = np.random.normal(loc=0, scale=1, size=5000)   # 대칭형(정규분포)
skewed_data = np.random.exponential(scale=1.5, size=5000)   # 오른쪽으로 긴 꼬리(지수분포)

def print_shape_metrics(data, name):
    mean = np.mean(data)          # 1차 적률: 평균 (위치)
    var = np.var(data)            # 2차 적률: 분산 (퍼짐)
    skew = stats.skew(data)       # 3차 적률: 왜도 (기울어짐)
    kurt = stats.kurtosis(data)   # 4차 적률: 첨도 (뾰족함/꼬리 두께, 정규분포 기준 초과값)
    print(f"=== [{name}] ===")
    print(f"1. 평균(Mean):  {mean:8.4f}  -> 분포의 중심 위치")
    print(f"2. 분산(Var):   {var:8.4f}  -> 퍼짐 정도")
    print(f"3. 왜도(Skew):  {skew:8.4f}  -> 좌우 비대칭도 (0=대칭, +면 오른쪽 꼬리)")
    print(f"4. 첨도(Kurt):  {kurt:8.4f}  -> 뾰족함/꼬리 두께 (0=정규분포와 비슷)\n")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, name, color in [
    (axes[0], normal_data, 'Symmetric (Normal)', 'skyblue'),
    (axes[1], skewed_data, 'Skewed (Exponential)', 'salmon'),
]:
    ax.hist(data, bins=40, density=True, color=color, alpha=0.8)
    ax.axvline(np.mean(data), color='red', linestyle='--', label=f'Mean={np.mean(data):.2f}')
    ax.set_title(f"{name}\nSkew={stats.skew(data):.2f} | Kurt={stats.kurtosis(data):.2f}")
    ax.legend()
plt.tight_layout()
plt.show()

print_shape_metrics(normal_data, "대칭형 분포 (Normal)")
print_shape_metrics(skewed_data, "비대칭형 분포 (Exponential)")


## 6. 영화 추천 시스템 — 협업 필터링(Collaborative Filtering)

"나랑 취향이 비슷한 사람이 재밌게 본 영화 중에 내가 안 본 영화"를 추천하는 방식이에요.
실제 MovieLens 데이터(`movies.dat`, `ratings.dat`, `users.dat`)로 하는 방식(`movie_systme.py`)을
작은 장난감 데이터로 그대로 재현해봅니다.


In [ ]:
# 6명의 친구가 6개 영화에 매긴 평점 (0 = 안 봄). 웹앱의 영화 추천 놀이터와 동일한 데이터
ratings = pd.DataFrame({
    '어벤져스':     [5, 5, 4, 0, 0, 0],
    '타이타닉':     [0, 0, 0, 5, 4, 5],
    '인터스텔라':   [5, 4, 5, 0, 0, 0],
    '라라랜드':     [0, 5, 0, 5, 5, 4],
    '기생충':       [4, 5, 4, 0, 5, 0],
    '인사이드아웃': [0, 0, 0, 4, 5, 5],
}, index=['철수', '민수', '현우', '영희', '지은', '수아'])

print("유저 x 영화 평점표:")
print(ratings)

# 유저들 간의 상관계수(취향 유사도) 행렬 - 실제 movie_systme.py의 pivot.T.corr()와 같은 방식
sim_matrix = ratings.T.corr()

def recommend_movie(user, n=2):
    # 나를 제외한 유저들과의 유사도를 내림차순 정렬해서 이웃 n명 고르기
    similarities = sim_matrix[user].drop(user).sort_values(ascending=False)
    neighbors = similarities.head(n)
    print(f"\n[{user}]와(과) 취향이 비슷한 친구 top {n}:")
    print(neighbors)

    # 내가 아직 안 본 영화(평점 0) 목록
    unseen_movies = ratings.columns[ratings.loc[user] == 0]

    # 이웃들이 5점 준 영화 중에서, 내가 안 본 영화만 추천
    recommend = set()
    for neighbor in neighbors.index:
        liked = ratings.columns[ratings.loc[neighbor] == 5]
        recommend |= set(liked) & set(unseen_movies)

    print(f"'{user}'님을 위한 추천 영화: {sorted(recommend) if recommend else '추천할 영화가 없어요'}")

recommend_movie('철수', n=2)
recommend_movie('영희', n=2)


## 7. 파이썬 기초 — 튜플(tuple)

`파이썬 기초문제.ipynb`의 071~080번 문제를 주석과 함께 정리했어요. 튜플은 "한 번 만들면 내용을
바꿀 수 없는(불변, immutable)" 자료형이라는 게 리스트와 가장 큰 차이예요.


In [ ]:
# 071 - 빈 튜플 만들기: 괄호만 쓰거나 tuple() 함수 사용
my_variable = ()
print(my_variable, type(my_variable))

# 072 - 콤마로 구분된 값들을 괄호로 묶으면 튜플이 됨
movie_rank = ('닥터 스트레인지', '스플릿', '럭키')
print(movie_rank)

# 073 - 요소가 1개인 튜플은 반드시 값 뒤에 콤마(,)를 붙여야 함
t = (1,)
print(t, type(t))          # (1,) <class 'tuple'>
print((1), type((1)))      # 콤마 없으면 그냥 정수 1! 튜플이 아니라서 주의

# 075 - 괄호 없이 콤마로만 나열해도 파이썬은 튜플로 인식함
t = 1, 2, 3, 4
print(type(t))

# 076 - 튜플 자체는 수정 불가(immutable)라서, 새 튜플을 만들어 변수를 다시 가리키게 함
t = ('a', 'b', 'c')
t = ('A', 'b', 'c')   # t[0]='A'처럼 직접 바꾸는 건 불가능, 통째로 재할당해야 함
print(t)

# 077~078 - 튜플 <-> 리스트 변환
interest = ('삼성전자', 'LG전자', 'SK Hynix')
interest_list = list(interest)          # 튜플 -> 리스트 (이제 수정 가능)
interest_tuple = tuple(interest_list)   # 리스트 -> 튜플 (다시 수정 불가능하게)
print(interest_list, type(interest_list))
print(interest_tuple, type(interest_tuple))

# 079 - 튜플 언패킹: 요소 개수와 변수 개수가 같아야 함
temp = ('apple', 'banana', 'cake')
a, b, c = temp
print(a, b, c)

# 080 - range(시작, 끝, 증가폭)으로 짝수만 뽑아 튜플로 만들기
even_numbers = tuple(range(2, 100, 2))
print(even_numbers)
